Step 1: Import Libraries & Detect Project Root

In [1]:
import os
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# Set project root
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if not os.path.exists(os.path.join(project_root, "data")):
    project_root = current_dir

# Paths to datasets
dataset1_path = os.path.join(project_root, "data", "raw", "cross_platform_dataset.csv")
dataset2_path = os.path.join(project_root, "data", "raw", "kaggle_dataset.csv")
dataset3_path = os.path.join(project_root, "data", "raw", "twitter.csv")

Step 2: Load Datasets

In [2]:
df1 = pd.read_csv(dataset1_path)  # Cross-Platform Raw
df2 = pd.read_csv(dataset2_path)  # Kaggle Twitter
df3 = pd.read_csv(dataset3_path)  # Twitter API Fetched

print("Datasets loaded successfully ✅")

Datasets loaded successfully ✅


Step 3: Harmonize Column Names & Labels

In [3]:
# Dataset 1
df1.rename(columns={"Text_Content":"Text_Content", "Platform":"Platform", "Target":"Label"}, inplace=True)
df1["Label"] = df1["Label"].apply(lambda x: "Cyberbullying" if x=="Cyberbullying" else "not_cyberbullying")

# Dataset 2 (Kaggle Twitter)
df2.rename(columns={"tweet_text":"Text_Content", "cyberbullying_type":"Label"}, inplace=True)
df2["Platform"] = "Twitter"
df2["Label"] = df2["Label"].apply(lambda x: "not_cyberbullying" if x=="not_cyberbullying" else "Cyberbullying")

# Dataset 3 (Twitter API Fetched)
df3.rename(columns={"text":"Text_Content", "label":"label", "platform":"Platform"}, inplace=True)
df3["Label"] = df3["label"].apply(lambda x: "Cyberbullying" if x==1 else "not_cyberbullying")

Step 4: Clean Text

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)                     # remove mentions
    text = re.sub(r"#\w+", "", text)                     # remove hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)             # remove special characters/numbers
    text = re.sub(r"\s+", " ", text).strip()            # remove extra spaces
    return text

# Apply cleaning
for d in [df1, df2, df3]:
    d["clean_text"] = d["Text_Content"].apply(clean_text)

Step 5: Encode Labels

In [5]:
label_mapping = {"Cyberbullying": 1, "not_cyberbullying": 0}

for d in [df1, df2, df3]:
    d["label_encoded"] = d["Label"].map(label_mapping)

Step 6: Split Training & Testing

In [6]:
# Training: Dataset 2
X_train, X_test, y_train, y_test = train_test_split(
    df2["clean_text"], df2["label_encoded"], test_size=0.2, random_state=42, stratify=df2["label_encoded"]
)

# Testing: Dataset 1
X_test_cross, y_test_cross = df1["clean_text"], df1["label_encoded"]

# Live Demo: Dataset 3
X_live_demo = df3["clean_text"]
y_live_demo = df3["label_encoded"]  # optional for reference

In [7]:
# Step 6.1: Save Train-Test Splits Separately

processed_folder = os.path.join(project_root, "data", "processed", "splits")
os.makedirs(processed_folder, exist_ok=True)

# Save training split
X_train.to_csv(os.path.join(processed_folder, "X_train.csv"), index=False)
y_train.to_csv(os.path.join(processed_folder, "y_train.csv"), index=False)

# Save test split (Twitter test)
X_test.to_csv(os.path.join(processed_folder, "X_test.csv"), index=False)
y_test.to_csv(os.path.join(processed_folder, "y_test.csv"), index=False)

# Save cross-platform test set
X_test_cross.to_csv(os.path.join(processed_folder, "X_test_cross.csv"), index=False)
y_test_cross.to_csv(os.path.join(processed_folder, "y_test_cross.csv"), index=False)

# Save live demo data
X_live_demo.to_csv(os.path.join(processed_folder, "X_live_demo.csv"), index=False)
y_live_demo.to_csv(os.path.join(processed_folder, "y_live_demo.csv"), index=False)

print("✅ Train, test, cross-platform, and live demo splits saved successfully")


✅ Train, test, cross-platform, and live demo splits saved successfully


Step 7: Save Processed Datasets

In [8]:
# Ensure processed folder exists
processed_folder = os.path.join(project_root, "data", "processed")
os.makedirs(processed_folder, exist_ok=True)

# Save Dataset 1 (Cross-Platform Raw)
df1_processed_path = os.path.join(processed_folder, "cross_platform_dataset_processed.csv")
df1.to_csv(df1_processed_path, index=False)

# Save Dataset 2 (Kaggle Twitter)
df2_processed_path = os.path.join(processed_folder, "kaggle_dataset_processed.csv")
df2.to_csv(df2_processed_path, index=False)

# Save Dataset 3 (Twitter API Fetched)
df3_processed_path = os.path.join(processed_folder, "twitter_api_processed.csv")
df3.to_csv(df3_processed_path, index=False)

print("✅ All processed datasets saved successfully in 'data/processed/'")

✅ All processed datasets saved successfully in 'data/processed/'


In [9]:
# Check missing value
print(df1[["clean_text", "label_encoded"]].isnull().sum())
print(df2[["clean_text", "label_encoded"]].isnull().sum())
print(df3[["clean_text", "label_encoded"]].isnull().sum())

clean_text       0
label_encoded    0
dtype: int64
clean_text       0
label_encoded    0
dtype: int64
clean_text       0
label_encoded    0
dtype: int64


In [10]:
# Check class balance
print(df1["label_encoded"].value_counts())
print(df2["label_encoded"].value_counts())
print(df3["label_encoded"].value_counts())

label_encoded
0    1173
1    1085
Name: count, dtype: int64
label_encoded
1    39747
0     7945
Name: count, dtype: int64
label_encoded
1    51
0    23
Name: count, dtype: int64


In [11]:
# Check a few samples
print(df1[["clean_text", "label_encoded"]].sample(5))
print(df2[["clean_text", "label_encoded"]].sample(5))
print(df3[["clean_text", "label_encoded"]].sample(5))

                           clean_text  label_encoded
1111                 i like your idea              0
1836             this is very helpful              0
1502                 i like your idea              0
690   nobody cares about your opinion              1
1186       great work on your project              0
                                              clean_text  label_encoded
31033  ya i get bullied a lot and make people feel re...              1
4189                      you should see his guilty look              0
24290                  i fucking love lychees in my ecig              1
25918                                              sq ft              1
7606   keep uploading cute photos of him makes me pin...              0
                                           clean_text  label_encoded
26  rt your mom would let you stare at her feet ju...              1
24  rt breaking ice just arrested this illegal ali...              1
72  rt bull shit bhagat you keep trying to 